### Example of working with VLM via the OpenRouter service

In [ ]:
import requests
import json

import base64
from dotenv import load_dotenv

import os
from pathlib import Path

from PIL import Image
from IPython.display import display

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import patches
from matplotlib.patches import FancyArrowPatch
from matplotlib.backends.backend_agg import FigureCanvasAgg

In [ ]:
def _repo_root(start: Path) -> Path:
    """Корень репозитория: вверх от каталога ноутбука/cwd, ищем .git или datasets/."""
    d = start.resolve()
    for p in [d, *d.parents]:
        if (p / ".git").exists():
            return p
    for p in [d, *d.parents]:
        if (p / "datasets").is_dir():
            return p
    return d


_nb = globals().get("__vsc_ipynb_file__")
_here = Path(_nb).resolve().parent if _nb else Path.cwd()
REPO_ROOT = _repo_root(_here)
os.chdir(REPO_ROOT)

# Ключ: metrics/open-router/env (от корня репо; после chdir тот же путь)
load_dotenv(REPO_ROOT / "metrics/open-router/env")

# env: OPEN_ROUTER_KEY=... (https://openrouter.ai/)
OPENROUTER_API_KEY = os.environ["OPEN_ROUTER_KEY"]

In [ ]:
# Пути от корня репозитория (после ячейки с os.chdir(REPO_ROOT)).
JSONL_PATH = "datasets/data_playground/PSG_json/test.jsonl"
ROOT_IMAGES = "datasets/frames/PSG_frames/test_images"
# OpenRouter model
MODEL = "openai/gpt-5.4-mini"

In [ ]:
# Function for encoding an image in base64
def encode_image_to_base64(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

In [ ]:
# Image display function in Jupyter Notebook
def show_image(filepath: str):
    image = Image.open(filepath)
    image.thumbnail((600, 600))
    display(image)

In [ ]:
# Function for accessing VLM via the OpenRouter service
def response_img(query: str, filepath: str, model: str):
    base64_image = encode_image_to_base64(filepath)

    body = {
        "model": model,
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": query},
                    {
                        "type": "image_url",
                        "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"},
                    },
                ],
            }
        ],
        "temperature": 0.00,
    }
    m = model.lower()
    # Only Qwen: otherwise the provider rejects the request (unknown field chat_template_kwargs)
    if "qwen" in m:
        body["chat_template_kwargs"] = {"enable_thinking": False}
        # Disable thinking via unified API OpenRouter (for Qwen this is ok)
        body["reasoning"] = {"effort": "none"}
    # For openai/* do not send reasoning.effort=none: OpenRouter responds with
    # «Reasoning is mandatory for this endpoint and cannot be disabled» (HTTP 400).
    response = requests.post(
        url="https://openrouter.ai/api/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {OPENROUTER_API_KEY}",
            "Content-Type": "application/json",
            "HTTP-Referer": "test",
            "X-Title": "test",
        },
        data=json.dumps(body),
    )
    data = response.json()
    if "choices" not in data:
        raise RuntimeError(
            f"OpenRouter: нет choices (HTTP {response.status_code}). Ответ: {json.dumps(data, ensure_ascii=False)[:4000]}"
        )
    return data

def openrouter_assistant_text(res: dict) -> str:
    """Text of the assistant's answer: string or list of blocks (OpenAI-compatible multimodal)."""
    msg = res["choices"][0]["message"]
    c = msg.get("content")
    if c is None:
        return ""
    if isinstance(c, str):
        return c
    if isinstance(c, list):
        out = []
        for block in c:
            if isinstance(block, dict) and block.get("type") == "text":
                out.append(block.get("text", ""))
            elif isinstance(block, str):
                out.append(block)
        return "\n".join(s for s in out if s).strip()
    return str(c)

In [ ]:
def read_all_samples(jsonl_path: str) -> list:
    """Reads all examples from the JSONL file."""
    samples = []
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            samples.append(json.loads(line))
    return samples


def _message_content_to_str(content) -> str:
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for block in content:
            if isinstance(block, dict) and block.get("type") == "text":
                parts.append(block.get("text", ""))
        return "\n".join(parts)
    return ""


def get_gpt_annotation(sample: dict) -> str:
    """Ground truth: ShareGPT (`conversations` / from=gpt) or MS Swift (`messages` / role=assistant)."""
    conv = sample.get("conversations", [])
    if conv:
        gpt_msgs = [c["value"] for c in conv if c.get("from") == "gpt"]
        if not gpt_msgs and conv:
            return conv[-1]["value"]
        return "\n".join(gpt_msgs)
    msgs = sample.get("messages", [])
    out = [_message_content_to_str(m.get("content")) for m in msgs if m.get("role") == "assistant"]
    return "\n".join(s for s in out if s).strip()


def get_prompt_text(sample: dict) -> str:
    """Prompt text: ShareGPT (not gpt) or MS Swift (all messages except assistant)."""
    conv = sample.get("conversations", [])
    if conv:
        user_msgs = [c["value"] for c in conv if c.get("from") != "gpt"]
        return "\n\n".join(user_msgs)
    msgs = sample.get("messages", [])
    parts = []
    for m in msgs:
        if m.get("role") == "assistant":
            continue
        s = _message_content_to_str(m.get("content"))
        if s:
            parts.append(s)
    return "\n\n".join(parts)


def get_image_path(sample: dict, root_images: str) -> str:
    """Path to the file: MS/Swift — list `images` (often absolute path); otherwise field `image` relative to root_images."""
    raw = sample.get("images")
    if isinstance(raw, list) and raw:
        p = str(raw[0]).strip()
        if not p:
            raise ValueError("Пустой images[0]")
        if os.path.isabs(p):
            return os.path.normpath(p)
        return os.path.normpath(os.path.join(root_images, p))
    rel = sample.get("image", "")
    if rel is None or not str(rel).strip():
        raise ValueError("No path to the image: expected keys 'images' (list) or 'image'")
    return os.path.normpath(os.path.join(root_images, str(rel).strip()))

In [ ]:
def parse_scene_graph_output(answer_text: str) -> dict:
    """
    Parses the output text of the model into a structured scene graph format.
    Expected format: obj[N]{id,name,x1,y1,x2,y2}: ... rel[M]{subj,pred,obj}: ...
    Returns a dictionary with keys "objects" and "relations".
    """
    lines = [line.rstrip("\n") for line in answer_text.splitlines()]
    lines = [line for line in lines if line.strip()]

    obj_header_idx = None
    rel_header_idx = None
    for i, line in enumerate(lines):
        stripped = line.strip()
        if stripped.startswith("obj[") and "{id,name,x1,y1,x2,y2}" in stripped:
            obj_header_idx = i
        if stripped.startswith("rel[") and "{subj,pred,obj}" in stripped:
            rel_header_idx = i

    if obj_header_idx is None or rel_header_idx is None:
        raise ValueError("Unable to find obj[...] or rel[...] headers in the model's response")

    obj_lines = lines[obj_header_idx + 1 : rel_header_idx]
    objects = []
    for raw_line in obj_lines:
        stripped = raw_line.strip()
        if not stripped:
            continue
        parts = [part.strip() for part in stripped.split(",")]
        if len(parts) != 6:
            continue
        idx_str, name, x1_str, y1_str, x2_str, y2_str = parts
        try:
            obj_id = int(idx_str)
            x1, y1 = int(x1_str), int(y1_str)
            x2, y2 = int(x2_str), int(y2_str)
        except ValueError:
            continue
        x1, x2 = sorted((x1, x2))
        y1, y2 = sorted((y1, y2))
        objects.append({"id": obj_id, "label": name, "bbox": [x1, y1, x2, y2]})

    rel_lines = lines[rel_header_idx + 1 :]
    relations = []
    for raw_line in rel_lines:
        stripped = raw_line.strip()
        if not stripped:
            continue
        parts = [part.strip() for part in stripped.split(",")]
        if len(parts) != 3:
            continue
        subj_str, pred, obj_str = parts
        try:
            subj_id, obj_id = int(subj_str), int(obj_str)
        except ValueError:
            continue
        relations.append({"sub": subj_id, "obj": obj_id, "pred": pred})

    return {"objects": objects, "relations": relations}

In [ ]:
def scene_graph_looks_like_1000_grid(scene_graph: dict, img_w: int, img_h: int) -> bool:
    """True, if the coordinates cannot be pixels of the frame (there is a value wider/higher than the image) — typically a 0..1000 grid for VLM."""
    obs = scene_graph.get("objects") or []
    if not obs:
        return False
    max_x = max_y = 0
    for obj in obs:
        x1, y1, x2, y2 = obj["bbox"]
        max_x = max(max_x, x1, x2)
        max_y = max(max_y, y1, y2)
    return max_x > img_w or max_y > img_h

def scene_graph_bboxes_to_image_pixels(
    scene_graph: dict,
    bbox_coord_space: tuple[int, int] | None,
    img_w: int,
    img_h: int,
) -> dict:
    """Scales bbox from a grid (cw×ch), e.g. 1000×1000, to pixels of the original image."""
    if bbox_coord_space is None:
        return scene_graph
    cw, ch = bbox_coord_space
    sx = img_w / float(cw)
    sy = img_h / float(ch)
    objects = []
    for obj in scene_graph["objects"]:
        x1, y1, x2, y2 = obj["bbox"]
        x1i = int(round(x1 * sx))
        y1i = int(round(y1 * sy))
        x2i = int(round(x2 * sx))
        y2i = int(round(y2 * sy))
        x1i, x2i = sorted((max(0, min(img_w, x1i)), max(0, min(img_w, x2i))))
        y1i, y2i = sorted((max(0, min(img_h, y1i)), max(0, min(img_h, y2i))))
        objects.append({**obj, "bbox": [x1i, y1i, x2i, y2i]})
    return {"objects": objects, "relations": list(scene_graph["relations"])}

def draw_scene_graph(
    image,
    answer_text: str,
    max_width: int = None,
    title: str = None,
    bbox_coord_space: tuple[int, int] | None = None,
) -> Image.Image:
    """
    Visualizes the scene graph on the image: bbox of objects and arrows with predicates.
    image: path to the image or PIL.Image.
    bbox_coord_space: (1000, 1000) — force the answer to be in this grid; None — auto:
        if any coordinate is greater than the width/height of the frame, the grid 1000×1000 is applied (Qwen, Gemini, …).
    """
    if isinstance(image, (str, bytes, bytearray)):
        img = Image.open(image).convert("RGB")
    else:
        img = image.convert("RGB")
    original_width, original_height = img.size
    scale = 1.0
    if max_width is not None and original_width > max_width:
        scale = max_width / float(original_width)
        img = img.resize((int(original_width * scale), int(original_height * scale)), Image.LANCZOS)
    display_width, display_height = img.size

    scene_graph = parse_scene_graph_output(answer_text)
    space = bbox_coord_space
    if space is None and scene_graph_looks_like_1000_grid(
        scene_graph, original_width, original_height
    ):
        space = (1000, 1000)
    scene_graph = scene_graph_bboxes_to_image_pixels(
        scene_graph, space, original_width, original_height
    )
    dpi = 100
    fig_width = display_width / dpi
    fig_height = display_height / dpi
    if title:
        title_height_inches = 0.5
        total_height = fig_height + title_height_inches
        fig = plt.figure(figsize=(fig_width, total_height), dpi=dpi)
        title_ax = fig.add_axes([0, (fig_height / total_height), 1, title_height_inches / total_height])
        title_ax.text(0.5, 0.5, title, fontsize=14, fontweight="bold", ha="center", va="center")
        title_ax.axis("off")
        ax = fig.add_axes([0, 0, 1, fig_height / total_height])
    else:
        fig = plt.figure(figsize=(fig_width, fig_height), dpi=dpi)
        ax = fig.add_axes([0, 0, 1, 1])
    ax.imshow(img)
    ax.axis("off")

    num_objects = len(scene_graph["objects"])
    palette = plt.cm.tab20(np.linspace(0, 1, max(20, num_objects)))
    centers = {}
    for i, obj in enumerate(scene_graph["objects"]):
        x1, y1, x2, y2 = obj["bbox"]
        x1_s = int(round(x1 * scale))
        y1_s = int(round(y1 * scale))
        x2_s = int(round(x2 * scale))
        y2_s = int(round(y2 * scale))
        box_w = max(1, x2_s - x1_s)
        box_h = max(1, y2_s - y1_s)
        color = palette[i % len(palette)]
        ax.add_patch(
            patches.Rectangle((x1_s, y1_s), box_w, box_h, linewidth=2, edgecolor=color, facecolor="none")
        )
        fontsize = 9
        if box_h < 14:
            fontsize = 7
        if box_h < 9:
            fontsize = 6
        ax.text(
            x1_s + 2, y1_s + 2, f"{obj['id']}: {obj['label']}",
            fontsize=fontsize, color="white", va="top",
            bbox=dict(facecolor=color, alpha=0.75, pad=0.6),
        )
        centers[obj["id"]] = ((x1_s + x2_s) / 2, (y1_s + y2_s) / 2)

    for relation in scene_graph["relations"]:
        subj_id, obj_id = relation["sub"], relation["obj"]
        predicate = relation["pred"]
        if subj_id in centers and obj_id in centers:
            sx, sy = centers[subj_id]
            ox, oy = centers[obj_id]
            ax.add_patch(
                FancyArrowPatch((sx, sy), (ox, oy), arrowstyle="->", mutation_scale=14,
                                color="yellow", linewidth=2, alpha=0.9)
            )
            mid_x, mid_y = (sx + ox) / 2, (sy + oy) / 2
            ax.text(mid_x, mid_y, predicate, fontsize=8, color="yellow",
                    bbox=dict(facecolor="black", alpha=0.55, boxstyle="round,pad=0.2"))

    canvas = FigureCanvasAgg(fig)
    canvas.draw()
    buffer = np.asarray(canvas.buffer_rgba())
    plt.close(fig)
    return Image.fromarray(buffer, mode="RGBA").convert("RGB")

def visualize_comparison(
    image_path: str,
    gt_text: str,
    pred_text: str,
    max_width: int = 800,
) -> Image.Image:
    """Left GT, right prediction. Bbox scale: auto (grid 1000×1000, if coordinates exceed the frame size)."""
    vis_gt = draw_scene_graph(image_path, gt_text, max_width=max_width, title="Ground Truth")
    vis_pred = draw_scene_graph(image_path, pred_text, max_width=max_width, title="Prediction")
    gt_width, gt_height = vis_gt.size
    pred_width, pred_height = vis_pred.size
    target_height = max(gt_height, pred_height)
    if gt_height != target_height:
        scale = target_height / gt_height
        vis_gt = vis_gt.resize((int(gt_width * scale), target_height), Image.LANCZOS)
        gt_width = int(gt_width * scale)
    if pred_height != target_height:
        scale = target_height / pred_height
        vis_pred = vis_pred.resize((int(pred_width * scale), target_height), Image.LANCZOS)
        pred_width = int(pred_width * scale)
    total_width = gt_width + pred_width
    combined = Image.new("RGB", (total_width, target_height), color="white")
    combined.paste(vis_gt, (0, 0))
    combined.paste(vis_pred, (gt_width, 0))
    return combined

### Comparison for a single frame (GT vs model)

 We load a single sample from the dataset, take an image and GT, request the model's response through OpenRouter, and draw a scene graph with Ground Truth on the left and prediction on the right.

In [ ]:
# Index from the example in JSONL (0 = first). Change and restart the cell to select a frame.

SAMPLE_IDX = 239

samples = read_all_samples(JSONL_PATH)
sample = samples[SAMPLE_IDX]
image_path = get_image_path(sample, ROOT_IMAGES)
gt_text = get_gpt_annotation(sample)

print("GT текст (scene graph):")
print(gt_text)
print()
show_image(image_path)

In [ ]:
# Uses the SAMPLE_IDX from the cell above (GT view). Restart that cell to change the frame.
samples = read_all_samples(JSONL_PATH)
sample = samples[SAMPLE_IDX]
image_path = get_image_path(sample, ROOT_IMAGES)
gt_text = get_gpt_annotation(sample)
# Prompt text for API (without placeholder <image>)
prompt_text = get_prompt_text(sample).replace("<image>", "").strip()

res = response_img(prompt_text, image_path, MODEL)
pred_text = openrouter_assistant_text(res)

# Visualization: left GT, right prediction (bbox 1000×1000 determined by heuristic for VLM)
comparison_img = visualize_comparison(image_path, gt_text, pred_text, max_width=800)
display(comparison_img)

In [ ]:
print(gt_text)

In [ ]:
print(pred_text)